In [51]:
import numpy as np
from scipy.stats import qmc

In [104]:
# sample training data
train_data = ["love games", "hate summer", "hate movie"]
train_labels = ["positive", "negative", "negative"]

In [105]:
# function to get sobol sequence
def get_sobol_sequqnce(dimension, n):
  sobol_sampler = qmc.Sobol(dimension)
  sobol_seqeunce = sobol_sampler.random(n)
  return sobol_seqeunce

In [ ]:
sobol_seq = get_sobol_sequqnce(1024,8)
# len(sobol_seq)
# sobol_seq.size
sobol_seq

In [107]:
# function to get hypervector encoding of each letter
def encoding_for_letter(sobol_seq, char):
  if char == " ":
        char = "\u0020"
  elif char == ".":
        char = "\u002E"

  index = ord(char) % len(sobol_seq)
  sobol_seq_value = sobol_seq[index]
  print(index)
  encoded_sobol_seq = np.where(sobol_seq_value > 0.5,1,-1)
  return encoded_sobol_seq

In [ ]:
# function to get a dictionary for each letter
def create_letter_hypervector_dict(sobol_seq):
    letter_hypervectors = {}
    for char in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ .":
        letter_hypervectors[char] = encoding_for_letter(sobol_seq, char)
    return letter_hypervectors

# Generate hypervectors for each letter
letter_hypervector_dict = create_letter_hypervector_dict(sobol_seq)
letter_hypervector_dict

In [109]:
# function to do xor between the hypervectors
def xor_hypervectors(hv1, hv2):
    return np.bitwise_xor(hv1, hv2)

In [110]:
# function to do bigram encdoing
def encode_bigram(bigram, letter_hypervector_dict):

    hv1 = letter_hypervector_dict[bigram[0]]
    hv2 = letter_hypervector_dict[bigram[1]]
    print(hv1, hv2)

    combined_hv = xor_hypervectors(hv1, hv2)

    return combined_hv

In [111]:
def train(data, labels):
    # dictionary to store class label and it's corresponding hypervectors
    class_hypervectors = {}

    for label in set(labels):
        class_hypervectors[label] = np.zeros(1024)

    for text, label in zip(data, labels):
        bigrams = [text[i:i+2] for i in range(len(text) - 1)]
        for bigram in bigrams:
            encoded_bigram = encode_bigram(bigram, letter_hypervector_dict)
            class_hypervectors[label] += encoded_bigram

    for label in class_hypervectors:
        hypervector = class_hypervectors[label]
        final_hypervector = np.where(hypervector > 0, 1, -1)
        class_hypervectors[label] = final_hypervector

    return class_hypervectors

In [ ]:
class_hypervectors = train(train_data, train_labels)
class_hypervectors

In [ ]:
from scipy.spatial.distance import cosine
def test(test_data, class_hypervectors):
    encoded_test_data = np.zeros(1024)
    # hypervector encoding the test data
    for char in test_data:
        sobol_value = letter_hypervector_dict[char]
        encoded_test_data += np.where(sobol_value > 0.5, 1, -1)


    similarities = {}
    for label, hypervector in class_hypervectors.items():
        similarities[label] = 1 - cosine(encoded_test_data, hypervector)

    return max(similarities, key=similarities.get)